In [ ]:
!pip install gradio


In [ ]:
# ============================================================
# RAW DATA → RFM FEATURE ENGINEERING
# ============================================================

import pandas as pd

# ============================================================
# STEP 1: LOAD RAW DATASET
# ============================================================
# This should be your ORIGINAL dataset (transactions / visits)
raw_data = pd.read_csv("raw_ecommerce_sample.csv")

print("Raw data loaded")
print("Shape:", raw_data.shape)

# ============================================================
# STEP 2: BASIC PREPROCESSING
# ============================================================

# Convert visit_date to datetime
raw_data['visit_date'] = pd.to_datetime(raw_data['visit_date'])

# ============================================================
# STEP 3: DEFINE REFERENCE DATE
# ============================================================
# Usually the latest date in the dataset
reference_date = raw_data['visit_date'].max()

print("Reference date:", reference_date.date())

# ============================================================
# STEP 4: CALCULATE RFM FEATURES
# ============================================================

# -------- RECENCY --------
last_visit = raw_data.groupby('customer_id')['visit_date'].max()
recency_days = (reference_date - last_visit).dt.days

# -------- FREQUENCY --------
# Number of unique sessions / orders per customer
frequency = raw_data.groupby('customer_id')['session_id'].nunique()

# -------- MONETARY --------
# Total amount spent by customer
monetary = raw_data.groupby('customer_id')['total_amount'].sum()

# ============================================================
# STEP 5: COMBINE INTO RFM DATAFRAME
# ============================================================

rfm_data = pd.DataFrame({
    'customer_id': frequency.index,
    'frequency': frequency.values,
    'monetary': monetary.values,
    'recency_days': recency_days.values
})

print("\nRFM feature engineering completed")
print(rfm_data.head())

# ============================================================
# STEP 6: SAVE ENGINEERED DATA
# ============================================================

rfm_data.to_csv("rfm_engineered_data.csv", index=False)

print("\n✅ RFM engineered data saved as rfm_engineered_data.csv")
print("This file is ready for your FINAL inference code")

In [1]:
import gradio as gr
import pandas as pd
import joblib
import os

# ===============================
# LOAD MODELS
# ===============================
churn_model = joblib.load("churn_random_forest_model.pkl")
repeat_model = joblib.load("repeat_purchase_logistic_model.pkl")

engineered_data = None
prediction_data = None

# ===============================
# STATUS LOGIC
# ===============================
def get_status(churn_prob, repeat_prob, monetary):
    if churn_prob > 0.7:
        return "Churned"
    elif churn_prob > 0.4:
        return "At Risk"
    elif repeat_prob > 0.7:
        return "Loyal"
    elif monetary > 2000:
        return "High Value"
    else:
        return "Regular"

# ===============================
# RUN INFERENCE
# ===============================
def run_inference(df):
    df = df.copy()

    X_churn = df[['frequency','monetary']]
    X_rp = df[['recency_days','monetary']]

    df['churn_probability'] = churn_model.predict_proba(X_churn)[:,1]
    df['repeat_probability'] = repeat_model.predict_proba(X_rp)[:,1]

    df['status'] = df.apply(
        lambda r: get_status(
            r["churn_probability"],
            r["repeat_probability"],
            r["monetary"]), axis=1)

    return df

# ===============================
# SUMMARY CALCULATION
# ===============================
def calculate_summary(df):

    total = len(df)

    churn_pct = (df["churn_probability"] > 0.5).mean()*100
    repeat_pct = (df["repeat_probability"] > 0.5).mean()*100

    high_value_pct = (df["status"]=="High Value").mean()*100
    loyal_pct = (df["status"]=="Loyal").mean()*100
    risk_pct = (df["status"]=="At Risk").mean()*100
    churned_pct = (df["status"]=="Churned").mean()*100

    summary = f"""
Total Customers: {total}

Likely to Churn: {churn_pct:.1f}%
Likely to Repeat: {repeat_pct:.1f}%

High Value: {high_value_pct:.1f}%
Loyal: {loyal_pct:.1f}%
At Risk: {risk_pct:.1f}%
Churned: {churned_pct:.1f}%
"""
    return summary

# ===============================
# CSV UPLOAD
# ===============================
def upload_csv(file):

    global engineered_data, prediction_data

    engineered_data = pd.read_csv(file.name)
    prediction_data = run_inference(engineered_data)

    summary = calculate_summary(prediction_data)

    return prediction_data, summary

# ===============================
# SEARCH CUSTOMER
# ===============================
def search_customer(customer_id):

    global prediction_data

    if prediction_data is None:
        return "Upload CSV first","","",""

    customer = prediction_data[
        prediction_data["customer_id"] == int(customer_id)
    ]

    if customer.empty:
        return "Customer not found","","",""

    row = customer.iloc[0]

    return (
        row["status"],
        f"{row['churn_probability']:.2f}",
        f"{row['repeat_probability']:.2f}",
        f"₹{row['monetary']}"
    )

# ===============================
# SAVE & APPEND
# ===============================
def save_results():

    global prediction_data

    if prediction_data is None:
        return "No data"

    file = "prediction_results_log.csv"

    if os.path.exists(file):
        prediction_data.to_csv(file, mode="a", header=False, index=False)
    else:
        prediction_data.to_csv(file, index=False)

    return "Saved & appended successfully"

# ===============================
# UI
# ===============================
with gr.Blocks(title="Customer Behavior Dashboard") as demo:

    gr.Markdown("# Customer Behavior Dashboard")

    with gr.Row():
        file_input = gr.File(label="Upload Engineered CSV")
        upload_btn = gr.Button("Run Predictions")

    summary_box = gr.Textbox(label="Customer Summary", lines=8)
    output_table = gr.Dataframe(label="Prediction Output")

    upload_btn.click(upload_csv, file_input, [output_table, summary_box])

    gr.Markdown("## Search Customer")

    cust_id = gr.Textbox(label="Enter Customer ID")
    search_btn = gr.Button("Search")

    status = gr.Textbox(label="Status")
    churn = gr.Textbox(label="Churn Probability")
    repeat = gr.Textbox(label="Repeat Probability")
    money = gr.Textbox(label="Monetary")

    search_btn.click(
        search_customer,
        cust_id,
        [status, churn, repeat, money]
    )

    gr.Markdown("## Save Results")
    save_btn = gr.Button("Save & Append Results")
    save_msg = gr.Textbox()

    save_btn.click(save_results, None, save_msg)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d27ac26fb2cf0cdc2e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
